# OpenPlaque — Alternate-Series Left Coronary Origin Validation v1

This experiment searches the **other DICOM series from the same CCTA exam** for independent phase/reconstruction evidence about the unresolved left-coronary origin.

It automatically inventories and ranks the series, selects the most informative contrast coronary candidates, aligns them to frozen Series 7, runs ImageCAS-X CAS-Net on each, and adjudicates **distinct RCA vs left aortic contact clusters**. The frozen master is never modified.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Reuse/cache controls — intentionally immediately after Drive mount
REUSE_VALID_MODEL_PREDICTIONS = True
FORCE_EXTERNAL_INFERENCE = False

DRIVE_ROOT = "/content/drive/MyDrive/OpenPlaque"
DICOM_ROOT = "/content/drive/MyDrive/CCTA/DICOM/3221"
OUT = f"{DRIVE_ROOT}/Alternate_Series_Left_Coronary_Origin_Validation_v1"
LOCAL_WORKDIR = "/content/openplaque_alternate_series"
print("DICOM:", DICOM_ROOT)
print("Output:", OUT)

In [ ]:
import os, shutil, sys, subprocess, json, zipfile
from pathlib import Path

OPENPLAQUE_PIN = "1e58b849c215f3ddd8979d302e75817a9429dac9"
OPENPLAQUE_BRANCH = "alternate-series-left-coronary-origin-validation-from-main"
IMAGECAS_X_PIN = "dbc7343187adf45ca9306dc3460eebc70f92b204"

for p in ["/content/OpenPlaque", "/content/ImageCAS-X"]:
    if Path(p).exists():
        shutil.rmtree(p)

!git clone -q --branch {OPENPLAQUE_BRANCH} https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout -q {OPENPLAQUE_PIN}
!git clone -q https://github.com/kitbransby/ImageCAS-X.git /content/ImageCAS-X
!git -C /content/ImageCAS-X checkout -q {IMAGECAS_X_PIN}

%pip install -q /content/OpenPlaque
%pip install -q /content/ImageCAS-X
%pip install -q requests

for name in list(sys.modules):
    if name == "openplaque" or name.startswith("openplaque."):
        del sys.modules[name]

print("OpenPlaque pin:", subprocess.check_output(["git","-C","/content/OpenPlaque","rev-parse","HEAD"], text=True).strip())
print("ImageCAS-X pin:", subprocess.check_output(["git","-C","/content/ImageCAS-X","rev-parse","HEAD"], text=True).strip())

In [ ]:
# Synthetic tests before science
from openplaque.alternate_series_left_coronary_origin_validation_v1 import synthetic_self_test
print(synthetic_self_test())
!cd /content/OpenPlaque && pytest -q tests/test_alternate_series_left_coronary_origin_validation_v1.py

In [ ]:
from openplaque.alternate_series_left_coronary_origin_validation_v1 import prepare
from IPython.display import display
import pandas as pd

dicom_path = Path(DICOM_ROOT)
if not dicom_path.exists():
    raise FileNotFoundError(f"Verified CCTA DICOM path is not mounted: {dicom_path}")
child_dirs = [p for p in dicom_path.iterdir() if p.is_dir()]
print("DICOM root:", dicom_path)
print("Series folders visible:", len(child_dirs))
print("First series folders:", [p.name for p in sorted(child_dirs)[:10]])

prep = prepare(
    drive_root=DRIVE_ROOT,
    dicom_root=DICOM_ROOT,
    local_workdir=LOCAL_WORKDIR,
    output_dir=OUT,
)

print("Reference series:", prep["reference_series"].get("series_number"), prep["reference_series"].get("series_description"))
print("Selected alternate series:")
display(pd.DataFrame(prep["selected_series"])[["scan_id","role","series_number","series_description","phase_key"]])
display(pd.read_csv(Path(OUT)/"candidate_ranking.csv").head(12))

## Independent coronary-lumen model

The selected series are now fixed. The next cells run the same pretrained ImageCAS-X CAS-Net observer across Series 7 and every selected alternate series.

The model is used only as independent lumen evidence. A left-root result requires a **distinct aortic-contact cluster separated from the RCA ostium**, so the Series 7 false-positive mechanism cannot satisfy the left gate.

In [ ]:
import requests

def acquire_casnet_checkpoint(cache_dir):
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    existing = list(cache_dir.rglob("*cas*net*.pt")) + list(cache_dir.rglob("*cas*net*.pth"))
    if existing:
        return existing[0]

    rec = requests.get("https://zenodo.org/api/records/21887809", timeout=60)
    rec.raise_for_status()
    files = rec.json()["files"]

    def score(f):
        k = f["key"].lower()
        s = 0
        if "cas_net" in k or "cas-net" in k or "casnet" in k: s += 100
        elif "cas" in k: s += 40
        if "weight" in k or "pretrain" in k or "checkpoint" in k: s += 20
        if k.endswith((".pt",".pth")): s += 20
        if k.endswith((".zip",".tar.gz",".tgz")): s += 10
        return s

    for f in sorted(files, key=score, reverse=True):
        if score(f) <= 0:
            continue
        key = f["key"]
        target = cache_dir / key.replace("/", "_")
        if not target.exists():
            url = f.get("links",{}).get("content") or f.get("links",{}).get("self")
            print("Downloading:", key)
            with requests.get(url, stream=True, timeout=180) as r:
                r.raise_for_status()
                with open(target, "wb") as w:
                    for chunk in r.iter_content(8*1024*1024):
                        if chunk:
                            w.write(chunk)
        if target.suffix.lower() in (".pt",".pth") and "cas" in target.name.lower():
            return target
        if zipfile.is_zipfile(target):
            ex = cache_dir / (target.name + "_extracted")
            ex.mkdir(exist_ok=True)
            with zipfile.ZipFile(target) as z:
                z.extractall(ex)
            hits = [p for p in ex.rglob("*") if p.suffix.lower() in (".pt",".pth") and "cas" in p.name.lower()]
            if hits:
                return hits[0]

    raise RuntimeError("Could not identify CAS-Net checkpoint in Zenodo record 21887809")

weight_cache = Path(DRIVE_ROOT)/"Cache"/"ImageCAS_X_Pretrained_v1"
checkpoint = acquire_casnet_checkpoint(weight_cache)
print("CAS-Net checkpoint:", checkpoint)

In [ ]:
import SimpleITK as sitk
import torch

scan_ids = [r["scan_id"] for r in prep["selected_series"]]
persistent_predictions = Path(OUT)/"external_model_predictions"
persistent_predictions.mkdir(parents=True, exist_ok=True)

all_cached = all((persistent_predictions/f"{sid}.nii.gz").is_file() and (persistent_predictions/f"{sid}.nii.gz").stat().st_size > 0 for sid in scan_ids)

if REUSE_VALID_MODEL_PREDICTIONS and all_cached and not FORCE_EXTERNAL_INFERENCE:
    predictions_dir = persistent_predictions
    print("Reusing cached ImageCAS-X predictions:", scan_ids)
else:
    if not torch.cuda.is_available():
        raise RuntimeError("A GPU runtime is required for ImageCAS-X inference. Switch Colab to a GPU runtime and use Runtime -> Run all.")

    imagecas = Path("/content/ImageCAS-X")
    data_root = Path("/content/imagecas_x_alternate_series_data")
    results_root = Path("/content/imagecas_x_alternate_series_results")
    run_dir = results_root/"cas_net_openplaque"
    for d in [data_root/"volumes", data_root/"segmentations", data_root/"filelist", results_root, run_dir]:
        d.mkdir(parents=True, exist_ok=True)

    model_input_dir = Path(prep["model_input_dir"])
    for sid in scan_ids:
        src = model_input_dir/f"{sid}.img.mha"
        dst = data_root/"volumes"/f"{sid}.img.mha"
        shutil.copy2(src, dst)

        ref_img = sitk.ReadImage(str(dst))
        dummy = sitk.Image(ref_img.GetSize(), sitk.sitkUInt8)
        dummy.CopyInformation(ref_img)
        mpath = data_root/"segmentations"/f"{sid}.coronary.mha"
        writer = sitk.ImageFileWriter()
        writer.SetFileName(str(mpath))
        writer.SetUseCompression(False)
        writer.Execute(dummy)

    (data_root/"filelist"/"train.txt").write_text("")
    (data_root/"filelist"/"val.txt").write_text("")
    (data_root/"filelist"/"test.txt").write_text("\n".join(scan_ids) + "\n")
    (data_root/"filelist"/"exclude.txt").write_text("")

    cfg_path = imagecas/"configs"/"cas_net_openplaque_alt_series.json"
    cfg = json.loads((imagecas/"configs"/"cas_net.json").read_text())
    cfg.setdefault("data",{})["volume_suffix"] = ".img.mha"
    cfg.setdefault("data",{})["mask_suffix"] = ".coronary.mha"
    cfg.setdefault("data",{}).setdefault("params",{})["inference_batch_size"] = 1
    cfg.setdefault("training",{})["num_workers"] = 0
    cfg["model"]["checkpoint"] = str(checkpoint)
    cfg_path.write_text(json.dumps(cfg, indent=2))

    os.environ["ImageCAS_X_data_path"] = str(data_root)
    os.environ["ImageCAS_X_results_path"] = str(results_root)

    resample_cmd = [
        sys.executable, "-u", "-m", "utils.offline_resample_images_to_disk",
        "-c", str(cfg_path), "--workers", "1", "--overwrite",
    ]
    print("Building ImageCAS-X 0.5-mm caches for", len(scan_ids), "series...")
    proc = subprocess.Popen(resample_cmd, cwd=imagecas, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"ImageCAS-X offline resampling failed with exit code {rc}; traceback is printed above.")

    for sid in scan_ids:
        for p in [data_root/"volumes_resampled"/f"{sid}.npy", data_root/"segmentations_resampled"/f"{sid}.npy"]:
            if not p.is_file() or p.stat().st_size == 0:
                raise RuntimeError(f"Missing ImageCAS-X cache: {p}")

    cmd = [sys.executable, "-u", "-m", "inference", "-c", str(cfg_path), "-r", str(run_dir), "--overwrite"]
    print("Running CAS-Net on:", scan_ids)
    proc = subprocess.Popen(cmd, cwd=imagecas, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"ImageCAS-X inference failed with exit code {rc}; traceback is printed above.")

    local_predictions = run_dir/"predictions"
    for sid in scan_ids:
        src = local_predictions/f"{sid}.nii.gz"
        if not src.is_file() or src.stat().st_size == 0:
            raise RuntimeError(f"Missing prediction: {src}")
        shutil.copy2(src, persistent_predictions/f"{sid}.nii.gz")

    predictions_dir = persistent_predictions
    print("Predictions cached to Drive:", predictions_dir)

In [ ]:
from openplaque.alternate_series_left_coronary_origin_validation_v1 import analyze

summary = analyze(
    predictions_dir=str(predictions_dir),
    drive_root=DRIVE_ROOT,
    local_workdir=LOCAL_WORKDIR,
    output_dir=OUT,
)

print(json.dumps(summary["decision"], indent=2))
print("\nStatus:", summary["status"])

In [ ]:
# Compact quantitative review
display(pd.read_csv(Path(OUT)/"registration_summary.csv"))
display(pd.read_csv(Path(OUT)/"path_support_by_series.csv"))
display(pd.read_csv(Path(OUT)/"aorta_contact_clusters.csv"))
display(pd.read_csv(Path(OUT)/"gates_by_series.csv"))
display(pd.read_csv(Path(OUT)/"series_decisions.csv"))

In [ ]:
# Show automatically selected root-plane QC
from IPython.display import Image, display, HTML

for p in sorted(Path(OUT).glob("QC_*_root_planes.png")):
    print(p.name)
    display(Image(filename=str(p), width=1000))

display(Image(filename=str(Path(OUT)/"01_aortic_contact_cluster_distances.png"), width=900))
display(HTML((Path(OUT)/"OPENPLAQUE_ALTERNATE_SERIES_LEFT_CORONARY_ORIGIN_VALIDATION_V1_REPORT.html").read_text()))

In [ ]:
# Verify expected deliverables
expected = [
    "run_state.json", "summary.json", "decision.json", "input_provenance.json",
    "preparation.json", "series_inventory.csv", "candidate_ranking.csv",
    "registration_summary.csv", "path_support_by_series.csv",
    "aorta_contact_clusters.csv", "gates_by_series.csv", "series_decisions.csv",
    "01_aortic_contact_cluster_distances.png",
    "OPENPLAQUE_ALTERNATE_SERIES_LEFT_CORONARY_ORIGIN_VALIDATION_V1_REPORT.html",
    "OPENPLAQUE_ALTERNATE_SERIES_LEFT_CORONARY_ORIGIN_VALIDATION_V1_RESULTS.zip",
]
missing = [x for x in expected if not (Path(OUT)/x).exists()]
qc = sorted(Path(OUT).glob("QC_*_root_planes.png"))
if missing:
    raise RuntimeError("Missing outputs: " + str(missing))
if len(qc) < len(prep["selected_series"]):
    raise RuntimeError(f"Expected at least {len(prep['selected_series'])} QC images, found {len(qc)}")
print("COMPLETE")
print("QC images:", len(qc))
for x in expected:
    print(Path(OUT)/x)